# Day 5: Instruction tuning

The pretrained model continues text. Ask it a question and it writes more questions, or an article. Today it reads
conversations and learns to answer.

This notebook runs [`scripts/train_sft.py`](https://github.com/a-ramakrishnan/mini_llm/blob/main/scripts/train_sft.py).
It needs Day 4's run on your Google Drive (`mini_llm/runs/<preset>-stage2`).

## Set up

**Runtime → Change runtime type → a GPU.** A free T4 runs the mini path. The full runs were done on an L4 and an A100.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU (or better)"
print(torch.cuda.get_device_name(0))

Runs are saved to Google Drive, so a disconnected session loses nothing: run the notebook again and training carries
on from the last saved state. Token files are kept on the fast local disk and backed up to Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
WORK = "/content/drive/MyDrive/mini_llm"      # runs/ and a copy of the token files live here
os.makedirs(f"{WORK}/runs", exist_ok=True)
os.makedirs(f"{WORK}/data", exist_ok=True)
os.environ["MINI_LLM_DATA"] = "/content/data"  # token files are read from the local disk: Drive is too slow
os.environ["WORK"] = WORK

In [ ]:
%cd /content
!test -d mini_llm || git clone -q https://github.com/a-ramakrishnan/mini_llm.git
%cd /content/mini_llm
!git pull -q
!pip install -q tiktoken datasets

In [ ]:
PRESET = "mini"  # "mini": 29.9M parameters, the mini path. "full": the shipped 124M run.
os.environ["PRESET"] = PRESET

In [ ]:
BASE = f"{WORK}/runs/{PRESET}-stage2"
RUN = f"{WORK}/runs/{PRESET}-sft"
assert os.path.exists(f"{BASE}/best_model_params.pt"), "Run Day 4 first: this tunes its model"

## 1. Before: the base model, asked a question

In [ ]:
!python scripts/chat.py --complete "User: What is the water cycle?

Assistant:" --model "{BASE}" --preset greedy --max-tokens 80

## 2. The chat format and the loss mask

Conversations are written as plain text, so GPT-2's tokenizer needs no new tokens. The loss counts only the
assistant's tokens and the `<|endoftext|>` that ends each answer.

In [ ]:
!python days/day_05_instruction_tuning/show_mask.py

## 3. The data

[smol-smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk): conversations written for small
models. Each is packed whole into a window of the model's context; one that is too long is cut after the last
answer that fits.

In [ ]:
!python scripts/train_sft.py --preset {PRESET} --base "{BASE}" --prepare-only

## 4. Tune

One pass over the conversations. The peak learning rate is 2e-4, a sixth of pretraining's: the model already knows
the language and only has to learn a new shape for its output. Every `eval_every` updates the loss on held-out
conversations is measured, on the answers only, and the best weights are kept.

In [ ]:
!python scripts/train_sft.py --preset {PRESET} --base "{BASE}" --out "{RUN}"

## 5. The held-out answer loss

In [ ]:
import json, matplotlib.pyplot as plt
sft = json.load(open(f"{RUN}/model_config.json"))["sft"]
train, test = sft["history"]["train"], sft["history"]["test"]
plt.figure(figsize=(7, 4))
plt.plot([t[0] for t in train], [t[1] for t in train], alpha=0.4, label="train (one batch)")
plt.plot([t[0] for t in test], [t[1] for t in test], marker="o", label="held-out answers")
plt.xlabel("update"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title(f"held-out answer loss {sft['base_test_loss']:.3f} -> {sft['best_test_loss']:.3f}")
plt.show()

## 6. After: it answers, and stops

In [ ]:
!python scripts/chat.py "What is the water cycle?" --model "{RUN}"
!python scripts/chat.py "Give me three tips for staying focused while studying." --model "{RUN}"

Look at the answers closely. They have the right shape, and they tend to repeat themselves. Greedy answers from the
shipped model at this stage repeat 16% of their 4-token runs, and about a quarter of them never end.
**Day 6 fixes that with preference tuning.**